In [5]:
import requests
import pandas as pd
import time

all_levels = []
page = 1

while True:
    response = requests.get(
        "https://api.tuforums.com/v2/database/levels",
        params={"page": page, "limit": 500}, timeout=10
    )
    data = response.json()
    all_levels.extend(data["results"])

    print(f"Page {page} done | Total levels fetched so far: {len(all_levels)}")

    if not data["hasMore"]:
        break
    page += 1
    time.sleep(0.1)  # be polite to their server

print(f"Pulled {len(all_levels)} levels total")

Page 1 done | Total levels fetched so far: 100
Page 2 done | Total levels fetched so far: 200
Page 3 done | Total levels fetched so far: 300
Page 4 done | Total levels fetched so far: 400
Page 5 done | Total levels fetched so far: 500
Page 6 done | Total levels fetched so far: 600
Page 7 done | Total levels fetched so far: 700
Page 8 done | Total levels fetched so far: 800
Page 9 done | Total levels fetched so far: 900
Page 10 done | Total levels fetched so far: 1000
Page 11 done | Total levels fetched so far: 1100
Page 12 done | Total levels fetched so far: 1200
Page 13 done | Total levels fetched so far: 1300
Page 14 done | Total levels fetched so far: 1400
Page 15 done | Total levels fetched so far: 1500
Page 16 done | Total levels fetched so far: 1600
Page 17 done | Total levels fetched so far: 1700
Page 18 done | Total levels fetched so far: 1800
Page 19 done | Total levels fetched so far: 1900
Page 20 done | Total levels fetched so far: 2000
Page 21 done | Total levels fetched so

In [6]:
rated_levels = []


for level in all_levels:
    diff = level.get("difficulty")
    level_id = level["id"]
    level["dlLink"] = f"https://api.tuforums.com/v2/database/levels/{level_id}/level.adofai"
    if isinstance(diff, dict):
        tier = diff.get("name")
        tier_upper = tier.upper() if tier else "" # Removing Q Tiers
        # Skip junk tiers
        if tier and not tier_upper.startswith(("Q", "PQ", "GQ", "UQ")) and tier not in ["SPECIAL", "Censored", "Impossible", "Unranked"]:
            level["difficulty"] = tier
            level["tuforums_link"] = f"https://tuforums.com/levels/{level.get('id')}" # Link to TUForums ADOFAI chart
            ms = level.get("levelLengthInMs")
            level["levelLengthInMs"] = round(float(ms) / 1000, 1) if ms is not None else 0.0 # Calculates song duration
            tile_count = level.get("tilecount")
            tile_count_val = float(tile_count) if tile_count is not None else 1.0
            level["density"] = level["levelLengthInMs"] / tile_count_val # Chart density formula
            rated_levels.append(level)


In [7]:
import sqlite3

DB_PATH = "levels.db"

def get_db():
    con = sqlite3.connect(DB_PATH, timeout=10)
    con.row_factory = sqlite3.Row
    return con

In [8]:
if not rated_levels:    
    print("Error! rated_levels is empty!")
else:
    df = pd.DataFrame(rated_levels)
    schema_columns = [
        "id", "song", "creator", "difficulty", "tilecount",
        "levelLengthInMs", "bpm", "tuforums_link", "dlLink", "density"
    ]


    df_to_save = df[schema_columns]


    with get_db() as con:
        con.execute("""
            CREATE TABLE IF NOT EXISTS raw_levels (
                id INTEGER PRIMARY KEY,
                song TEXT,
                creator TEXT,
                difficulty REAL,
                tilecount INTEGER,
                levelLengthInMs INTEGER,
                bpm REAL,
                tuforums_link TEXT,
                dlLink TEXT,
                density INTEGER
            )    
        """)


        df_to_save.to_sql("raw_levels", con, if_exists="replace", index=False)
    print("Databased saved for deep learning.")


Databased saved for deep learning.
